# Denoising algorithm evaluation

## 1. Import libraries

In [36]:
import soundfile as sf
import numpy as np
import pandas as pd
from scipy.signal import detrend
from scipy.signal.windows import tukey
from scipy.ndimage import uniform_filter1d
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import random

from summary_plots import summary_plots
from evaluation import FS, RMS_WIN, hp15, insert_artifact, metrics, run_experiment
import remove_noise as rn
from remove_noise import remove_noise

pd.set_option('display.max_columns', 500)

## 2. Algorithm

In [ ]:
# Algorithm constants used by the figures (see summary_plots)
PARAMS = dict(G_FAC=rn.G_FAC, K=rn.K, Z_CLICK=rn.Z_CLICK, Z_TOLERANCE=rn.Z_TOLERANCE, AR_ORDER=rn.AR_ORDER,
              ISO_WIN=rn.ISO_WIN, ISOLATION=rn.ISOLATION, ISOLATION_IMPULSIVE=rn.ISOLATION_IMPULSIVE,
              ISOLATION_ALIGNED=rn.ISOLATION_ALIGNED, MAX_MISALIGN=rn.MAX_MISALIGN)

## 3. Example

In [18]:
file = "9728"
wav_file = fr"..\dataset\dataset\audio\{file}.wav"

signal, fs = sf.read(wav_file)
clean_signal, details = remove_noise(signal, fs, return_details=True)

summary_plots(signal, fs, details, PARAMS)

## 4. Selection of signals

### 4.1. Selection of clean signals

**Selection criteria:**
- 10 signals per each label ("Normal", "Wheeze", "Rhonchi", "Stridor")
- Only 1 fragment per patient
- No clicks, scrapes, or impacts audible or visible in the spectrogram.
- Go through the candidates in the order in which they appear

In [19]:
clean_signals = {
    "Normal": [19,42,86,113,129,278,334,348,382,403],
    "Wheeze": [192,309,859,1109,1339,1729,1964,2233,2601,2654],
    "Rhonchi": [2,4272,4622,4855,5372,7392,7830,8094,8499,9828],
    "Stridor": [4108,9423,9735,14033,14505,17812,23218,15817],
}

In [20]:
clean_dict = {}
count = 0
for label, ids in clean_signals.items():
    for id in ids:
        wav_file = fr"..\dataset\dataset\audio\{id}.wav"

        clean_dict[count] = {}
        clean_dict[count]["signal"], _ = sf.read(wav_file)
        clean_dict[count]["id"] = id
        clean_dict[count]["label"] = label
        
        count += 1

### 4.2. Selection of artifacts

**Selection criteria (artifacts):**

- 40 artifacts in total
- At most 1 artifact per patient
- Patients not used in the clean signal set
- The artifact must be clearly visible
- Fully contained in the fragment: not touching the beginning or the end
- Go through the fragments in a fixed pseudo-random order (seed = 42)
- If there are a lot of artifacts of the same type, discard them

In [21]:
artifacts = [
    {"id": 22260, "t_ini": 1.15, "t_fin": 1.19},
    {"id": 22115, "t_ini": 1.01, "t_fin": 1.08},
    {"id": 5085,  "t_ini": 0.18, "t_fin": 0.24},
    {"id": 4999,  "t_ini": 1.53, "t_fin": 1.58},
    {"id": 16489, "t_ini": 0.54, "t_fin": 0.57},
    {"id": 2183,  "t_ini": 0.95, "t_fin": 1.00},
    {"id": 4303,  "t_ini": 0.51, "t_fin": 0.61},
    {"id": 15159, "t_ini": 1.27, "t_fin": 1.34},
    {"id": 13234, "t_ini": 0.30, "t_fin": 0.33},
    {"id": 984,   "t_ini": 0.96, "t_fin": 0.99},
    {"id": 22005, "t_ini": 0.38, "t_fin": 0.45},
    {"id": 924,   "t_ini": 0.74, "t_fin": 0.80},
    {"id": 16375, "t_ini": 1.48, "t_fin": 1.55},
    {"id": 7882,  "t_ini": 0.39, "t_fin": 0.44},
    {"id": 1673,  "t_ini": 0.11, "t_fin": 0.16},
    {"id": 22951, "t_ini": 0.15, "t_fin": 0.19},
    {"id": 21671, "t_ini": 1.65, "t_fin": 1.75},
    {"id": 20260, "t_ini": 1.60, "t_fin": 1.65},
    {"id": 5132,  "t_ini": 1.22, "t_fin": 1.38},
    {"id": 852,   "t_ini": 0.75, "t_fin": 0.86},
    {"id": 16814, "t_ini": 0.06, "t_fin": 0.11},
    {"id": 9135,  "t_ini": 1.05, "t_fin": 1.09},
    {"id": 17690, "t_ini": 1.24, "t_fin": 1.29},
    {"id": 23533, "t_ini": 1.35, "t_fin": 1.47},
    {"id": 2512,  "t_ini": 1.67, "t_fin": 1.74},
    {"id": 22895, "t_ini": 1.01, "t_fin": 1.08},
    {"id": 5963,  "t_ini": 0.06, "t_fin": 0.40},
    {"id": 9009,  "t_ini": 0.26, "t_fin": 0.32},
    {"id": 7577,  "t_ini": 0.50, "t_fin": 0.56},
    {"id": 14131, "t_ini": 0.36, "t_fin": 0.44},
    {"id": 23570, "t_ini": 0.80, "t_fin": 0.87},
    {"id": 20493, "t_ini": 1.04, "t_fin": 1.22},
    {"id": 15420, "t_ini": 0.10, "t_fin": 0.15},
    {"id": 7458,  "t_ini": 1.44, "t_fin": 1.48},
    {"id": 9793,  "t_ini": 0.79, "t_fin": 0.87},
    {"id": 22175, "t_ini": 1.33, "t_fin": 1.36},
    {"id": 22497, "t_ini": 0.66, "t_fin": 0.76},
    {"id": 14269, "t_ini": 0.24, "t_fin": 0.31},
    {"id": 14166, "t_ini": 0.76, "t_fin": 0.79},
    {"id": 22699, "t_ini": 1.00, "t_fin": 1.10},
]

In [22]:
artifact_dict = {}
for i, artifact in enumerate(artifacts):
    wav_file = fr"..\dataset\dataset\audio\{artifact["id"]}.wav"

    full_signal, fs = sf.read(wav_file)
    j_ini = int(round((artifact["t_ini"]-0.005) * fs))
    j_fin = int(round((artifact["t_fin"]+0.005) * fs))

    artifact_x = full_signal[j_ini:j_fin]
    artifact_pre = detrend(artifact_x, type="constant")

    alpha_tukey = 0.01*fs / len(artifact_pre)
    win = tukey(len(artifact_pre), alpha=alpha_tukey)
    artifact_pre = artifact_pre * win

    artifact_dict[i] = artifact_pre
    

## 5. Experiment 1: Insertion of a single artifact

### 5.1. Generation of mixtures

- Each clean signal is mixed with each artifact (that fits in it) at `N_REPS` random positions and at each SNR of `SNR_LIST`.
- **Control:** each clean signal is also processed without any artifact (`art_id` and `snr_db` = `"Control"`, no position or mixture information). It measures what the algorithm does to a signal without artifacts (false clicks, signal changed).

In [ ]:
SNR_LIST = [-20, -10, -5, 0, 5]  # SNR list to test (locals, in dB)
N_REPS = 5                       # Number of random positions per pair
EDGE_DIST = int(0.1 * FS)        # Minimum distance of the artifact to the edges (samples)
SEED = 42                        # Random seed for reproducibility

In [24]:
def get_mixes():
    random.seed(SEED)

    mixes_info = []
    count = 0
    for i, clean in clean_dict.items():
        clean_signal = clean["signal"]
        for j, artifact in artifact_dict.items():
            fit_margin = len(clean_signal) - len(artifact) - 2*EDGE_DIST
            if fit_margin > 0:
                for k in range(N_REPS):
                    k0 = EDGE_DIST + random.randint(0, fit_margin)
                    for snr in SNR_LIST:
                        _, info = insert_artifact(clean_signal, artifact, k0, snr)
                        mixes_info.append({
                            "id": count,
                            "clean_id": i,
                            "clean_file": clean["id"],
                            "label": clean["label"],
                            "art_id": j,
                            "rep": k,
                            "snr_db": snr,
                            **info
                        })
                        count += 1

    # Control: each clean signal without any artifact (no position, no mixture information)
    for i, clean in clean_dict.items():
        mixes_info.append({
            "id": count,
            "clean_id": i,
            "clean_file": clean["id"],
            "label": clean["label"],
            "art_id": "Control",
            "snr_db": "Control",
        })
        count += 1

    exp_df = pd.DataFrame(mixes_info)
    exp_df = exp_df.astype({"rep": "Int64", "k0": "Int64", "k1": "Int64"})  # NaN in the control rows
    exp_df.drop_duplicates(("clean_id", "art_id", "rep", "snr_db"))

    return exp_df

exp_df = get_mixes()
exp_df

,id,clean_id,clean_file,label,art_id,rep,snr_db,k0,k1,t0_s,art_duration_ms,g,p_local_rel_db,snr_global_db
0,0,0,19,Normal,0,0,-20,2224,2424,0.556,50.0,4.360387,1.361787,-4.563421
1,1,0,19,Normal,0,0,-10,2224,2424,0.556,50.0,1.378875,1.361787,5.436579
2,2,0,19,Normal,0,0,-5,2224,2424,0.556,50.0,0.775399,1.361787,10.436579
3,3,0,19,Normal,0,0,0,2224,2424,0.556,50.0,0.436039,1.361787,15.436579
4,4,0,19,Normal,0,0,5,2224,2424,0.556,50.0,0.245203,1.361787,20.436579
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37958,37958,33,14033,Stridor,Control,<NA>,Control,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN
37959,37959,34,14505,Stridor,Control,<NA>,Control,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN
37960,37960,35,17812,Stridor,Control,<NA>,Control,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN
37961,37961,36,23218,Stridor,Control,<NA>,Control,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN


In [25]:
ej = exp_df[exp_df.art_id != "Control"].sample(1, random_state=SEED).iloc[0]
filas_ej = exp_df[(exp_df.clean_id == ej.clean_id) & (exp_df.art_id == ej.art_id) & (exp_df.rep == ej.rep)]

clean = clean_dict[ej.clean_id]["signal"]
artifact = artifact_dict[ej.art_id]
k0 = ej.k0
t = np.arange(len(clean)) / FS

fig = make_subplots(
    rows=len(filas_ej),
    cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{f.snr_db} dB" for _, f in filas_ej.iterrows()],
    vertical_spacing=0.03,
)

for i, (_, f) in enumerate(filas_ej.iterrows(), start=1):
    snr = f.snr_db
    mixed, _ = insert_artifact(clean, artifact, k0, snr)

    fig.add_trace(
        go.Scatter(
            x=t, y=mixed,
            mode="lines",
            name="Signal with artifact",
            line=dict(color="darkorange", width=0.8),
            legendgroup="mixed",
            showlegend=(i == 1),
        ),
        row=i, col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=t, y=clean,
            mode="lines",
            name="Clean signal",
            line=dict(color="black", width=0.8),
            legendgroup="clean",
            showlegend=(i == 1),
        ),
        row=i, col=1,
    )

    fig.add_vrect(
        x0=f.k0 / FS,
        x1=f.k1 / FS,
        fillcolor="blue",
        opacity=0.15,
        line_width=0,
        row=i, col=1,
    )


fig.update_layout(
    title=(
        f"{ej.label} {ej.clean_id} ({ej.clean_file}) + "
        f"Artifact {ej.art_id} "
        f"(Relative Local Power = {ej.p_local_rel_db:.1f} dB)"
    ),
    height=1000,
    width=1200,
    template="plotly_white",
    hovermode="x unified",
)

fig.update_xaxes(title_text="Time (s)", row=len(filas_ej), title_font=dict(size=14))
fig.add_annotation(
    text="Amplitude (A.U.)",
    xref="paper", yref="paper",
    x=-0.08, y=0.5,
    showarrow=False,
    textangle=-90,
    font=dict(size=14),
)

fig.show()

### 5.2. Processing & Metrics

In [27]:
f = exp_df[(exp_df.snr_db == -5)].sample(5, random_state=SEED+8).iloc[0]      # una mezcla cualquiera a -5 dB

clean = clean_dict[f["clean_id"]]["signal"]
art = artifact_dict[f["art_id"]]
k0 = int(f["k0"])
k1 = k0 + len(art)

mixed, _ = insert_artifact(clean, art, k0, f["snr_db"])     # reconstruir la mezcla
out, det = remove_noise(mixed, FS, return_details=True)  # aplicar el algoritmo

ref = hp15(clean)       # limpia > 15 Hz
mix = hp15(mixed)       # mezcla > 15 Hz

m = metrics(ref, mix, out, k0, k1, det["click_span"])
print(pd.Series(m))

t = np.arange(len(ref)) / FS
z = slice(None)   # señal completa

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
)

# Subplot 1: señales
for y, name, color in [
    (mix[z], "Mezcla", "darkorange"),
    (out[z], "Salida", "royalblue"),
    (ref[z], "Limpia", "black"),
]:
    fig.add_trace(
        go.Scatter(
            x=t[z],
            y=y,
            mode="lines",
            name=name,
            line=dict(color=color, width=1),
        ),
        row=1, col=1,
    )

# Subplot 2: RMS
rms = lambda x: np.sqrt(uniform_filter1d(x**2, RMS_WIN))

for y, name, color in [
    (rms(mix)[z], "RMS mezcla", "darkorange"),
    (rms(out)[z], "RMS salida", "royalblue"),
    (rms(ref)[z], "RMS limpia", "black"),
]:
    fig.add_trace(
        go.Scatter(
            x=t[z],
            y=y,
            mode="lines",
            name=name,
            line=dict(color=color, width=1),
        ),
        row=2, col=1,
    )

# Zona del artefacto en ambos subplots
for row in (1, 2):
    fig.add_vrect(
        x0=k0 / FS,
        x1=k1 / FS,
        fillcolor="gray",
        opacity=0.2,
        line_width=0,
        layer="below",
        row=row, col=1,
    )

# Título y ejes
fig.update_layout(
    title=(
        f"{f['label']} {f['clean_file']} + artefacto {f['art_id']} "
        f"a {f['snr_db']} dB | "
        f"ΔSNR = {m['dsnr_db']:.1f} dB, "
        f"NMSE fuera = {m['nmse_out_zone_db']:.1f} dB"
    ),
    template="plotly_white",
    height=650,
    width=1200,
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
    ),
)

fig.update_yaxes(title_text="Amplitud (U.A.)", row=1, col=1)
fig.update_yaxes(title_text="RMS (U.A.)", row=2, col=1)
fig.update_xaxes(title_text="Tiempo (s)", row=2, col=1)

fig.show()
summary_plots(mixed, FS, det, PARAMS)

detected                      True
n_false                          0
snr_in_db                 -4.99892
snr_out_db                 9.69249
dsnr_db                   14.69141
attenuated                    True
nmse_out_zone_db        -16.741412
err_rms_max_loc_mix      165.24423
err_rms_max_loc_out       0.093466
err_rms_mean_loc_mix     50.893382
err_rms_mean_loc_out     -2.917039
err_rms_max_glob_mix     165.24423
err_rms_max_glob_out      0.093466
err_rms_mean_glob_mix    24.040868
err_rms_mean_glob_out    -3.578508
dtype: object


### 5.3. Execution of the experiment

`run_experiment` (see [`evaluation.py`](evaluation.py)) processes every mixture (and every control) in parallel with `multiprocessing` (one worker process per CPU core; `processes` to change it) and saves the metrics in `results/<csv_file>`.

In [ ]:
results_df = run_experiment(exp_df, "experiment_1.csv", clean_dict, artifact_dict)

### 5.4. Results

In [34]:
results_df = pd.read_csv("results/experiment_1.csv", low_memory=False)
results_df

,id,clean_id,clean_file,label,art_id,rep,snr_db,k0,k1,t0_s,art_duration_ms,g,p_local_rel_db,snr_global_db,detected,n_false,snr_in_db,snr_out_db,dsnr_db,attenuated,nmse_out_zone_db,err_rms_max_loc_mix,err_rms_max_loc_out,err_rms_mean_loc_mix,err_rms_mean_loc_out,err_rms_max_glob_mix,err_rms_max_glob_out,err_rms_mean_glob_mix,err_rms_mean_glob_out
0,0,0,19,Normal,0,0.0,-20,2224.0,2424.0,0.556,50.0,4.360387,1.361787,-4.563421,True,0,-19.999853,-2.705278,1.729457e+01,True,-49.119718,816.187136,-1.837039,441.242426,0.795917,505.184741,-1.934770e-04,24.378152,3.834876e-02
1,1,0,19,Normal,0,0.0,-10,2224.0,2424.0,0.556,50.0,1.378875,1.361787,5.436579,False,0,-9.999853,-9.999853,1.776357e-15,False,-49.956480,211.260109,211.260109,111.007645,111.007645,105.601958,1.056020e+02,6.132999,6.126937e+00
2,2,0,19,Normal,0,0.0,-5,2224.0,2424.0,0.556,50.0,0.775399,1.361787,10.436579,False,0,-4.999853,-4.999853,2.664535e-15,False,-50.027162,96.734328,96.734328,51.054306,51.054306,29.952287,2.995229e+01,2.820668,2.814606e+00
3,3,0,19,Normal,0,0.0,0,2224.0,2424.0,0.556,50.0,0.436039,1.361787,15.436579,False,0,0.000147,0.000147,1.928606e-15,False,-50.049755,40.754988,40.754988,21.911893,21.911893,-0.000019,-1.936467e-05,1.210597,1.204535e+00
4,4,0,19,Normal,0,0.0,5,2224.0,2424.0,0.556,50.0,0.245203,1.361787,20.436579,False,0,5.000147,5.000147,3.552714e-15,False,-50.056925,16.256866,16.256866,8.977206,8.977206,-0.000011,-1.089002e-05,0.495977,4.899147e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37958,37958,33,14033,Stridor,Control,NaN,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,-312.346967,NaN,NaN,NaN,NaN,0.000000,-2.146893e-14,0.000000,-1.693728e-12
37959,37959,34,14505,Stridor,Control,NaN,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,-54.991803,NaN,NaN,NaN,NaN,0.000000,-1.539049e-14,0.000000,-4.303864e-03
37960,37960,35,17812,Stridor,Control,NaN,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,-74.983649,NaN,NaN,NaN,NaN,0.000000,-2.616836e-14,0.000000,4.221891e-05
37961,37961,36,23218,Stridor,Control,NaN,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,-312.018659,NaN,NaN,NaN,NaN,0.000000,-1.293224e-14,0.000000,-1.884035e-13


#### 5.4.1. Main figures